# Final Sample Selection

This notebook converts the oversized positive candidate pool into the final
sample for manual annotation.

The positive sampling design contains:

- three target classes: human, animal and vehicle;
- seven positive MegaDetector confidence bands;
- four time windows;
- a final target of 20 class-records per sampling cell.

Relative bounding-box size, geographic group and season are used as diversity
balancing variables. They are not treated as additional formal sampling
strata.

The notebook will later add a separately sampled zero-confidence group and
produce a deduplicated manual-annotation queue.

In [2]:
from pathlib import Path

import numpy as np
import pandas as pd

In [3]:
INPUT_DIR = Path("sampling_outputs")
OUTPUT_DIR = Path("sampling_outputs")

POSITIVE_CANDIDATES_PATH = (
    INPUT_DIR
    / "positive_candidate_records_with_relative_bbox.csv"
)

BBOX_THRESHOLDS_PATH = (
    OUTPUT_DIR
    / "relative_bbox_group_thresholds.csv"
)

FINAL_POSITIVE_RECORDS_PATH = (
    OUTPUT_DIR
    / "final_positive_class_records.csv"
)

FINAL_POSITIVE_QUEUE_PATH = (
    OUTPUT_DIR
    / "final_positive_image_queue.csv"
)

RANDOM_SEED = 42
FINAL_PER_CELL = 20

TARGET_CLASSES = [
    "human",
    "animal",
    "vehicle"
]

CELL_COLUMNS = [
    "target_class",
    "sampling_confidence_band",
    "sampling_time_window"
]

In [4]:
candidate_records = pd.read_csv(
    POSITIVE_CANDIDATES_PATH,
    low_memory=False
)

print(
    "Total positive candidate records:",
    f"{len(candidate_records):,}"
)

print(
    "Columns:",
    len(candidate_records.columns)
)

print(
    "Relative bbox validity dtype:",
    candidate_records[
        "relative_bbox_area_valid"
    ].dtype
)

Total positive candidate records: 3,360
Columns: 77
Relative bbox validity dtype: bool


In [5]:
validity_values = (
    candidate_records[
        "relative_bbox_area_valid"
    ]
    .astype(str)
    .str.strip()
    .str.lower()
)

candidate_records[
    "eligible_for_final_selection"
] = validity_values.eq("true")

eligible_candidates = candidate_records.loc[
    candidate_records[
        "eligible_for_final_selection"
    ]
].copy()

print(
    "Eligible positive candidates:",
    f"{len(eligible_candidates):,}"
)

print(
    "Excluded positive candidates:",
    f"{(
        ~candidate_records[
            'eligible_for_final_selection'
        ]
    ).sum():,}"
)

Eligible positive candidates: 3,351
Excluded positive candidates: 9


In [6]:
eligible_class_counts = (
    eligible_candidates[
        "target_class"
    ]
    .value_counts()
    .reindex(TARGET_CLASSES)
    .rename_axis("target_class")
    .reset_index(name="eligible_candidates")
)

display(eligible_class_counts)

,target_class,eligible_candidates
0,human,1117
1,animal,1118
2,vehicle,1116


In [7]:
eligible_cell_counts = (
    eligible_candidates
    .groupby(
        CELL_COLUMNS,
        observed=True
    )
    .size()
    .rename("eligible_candidates")
    .reset_index()
)

print(
    "Number of positive sampling cells:",
    len(eligible_cell_counts)
)

print(
    "Minimum eligible records in one cell:",
    int(
        eligible_cell_counts[
            "eligible_candidates"
        ].min()
    )
)

print(
    "Maximum eligible records in one cell:",
    int(
        eligible_cell_counts[
            "eligible_candidates"
        ].max()
    )
)

assert len(candidate_records) == 3360
assert len(eligible_candidates) == 3351
assert len(eligible_cell_counts) == 84
assert eligible_cell_counts[
    "eligible_candidates"
].ge(FINAL_PER_CELL).all()

print("Positive candidate loading passed")

Number of positive sampling cells: 84
Minimum eligible records in one cell: 39
Maximum eligible records in one cell: 40
Positive candidate loading passed


## Relative bounding-box size groups

Eligible positive candidates are divided into four relative bounding-box size
groups using class-specific quartiles.

Quartile thresholds are calculated separately for human, animal and vehicle
because their bounding-box area distributions differ.

These categories are used only to preserve visual diversity during final
selection. They are not formal sampling strata or measures of physical object
size.

In [8]:
eligible_candidates[
    "relative_bbox_area"
] = pd.to_numeric(
    eligible_candidates[
        "relative_bbox_area"
    ],
    errors="coerce"
)

assert eligible_candidates[
    "relative_bbox_area"
].notna().all()

assert eligible_candidates[
    "relative_bbox_area"
].gt(0).all()

assert eligible_candidates[
    "relative_bbox_area"
].le(1).all()

BBOX_GROUP_LABELS = [
    "very_small",
    "small",
    "medium",
    "large"
]

eligible_candidates[
    "relative_bbox_group"
] = pd.NA

bbox_threshold_rows = []

for target_class in TARGET_CLASSES:
    class_mask = eligible_candidates[
        "target_class"
    ].eq(target_class)

    class_bbox_areas = eligible_candidates.loc[
        class_mask,
        "relative_bbox_area"
    ]

    q25 = class_bbox_areas.quantile(0.25)
    q50 = class_bbox_areas.quantile(0.50)
    q75 = class_bbox_areas.quantile(0.75)

    assert q25 < q50 < q75

    bbox_threshold_rows.append({
        "target_class": target_class,
        "q25": q25,
        "q50": q50,
        "q75": q75
    })

    bbox_groups = pd.cut(
        class_bbox_areas,
        bins=[
            -np.inf,
            q25,
            q50,
            q75,
            np.inf
        ],
        labels=BBOX_GROUP_LABELS,
        include_lowest=True
    )

    eligible_candidates.loc[
        class_mask,
        "relative_bbox_group"
    ] = bbox_groups.astype("string")

bbox_group_thresholds = pd.DataFrame(
    bbox_threshold_rows
)

display(bbox_group_thresholds)

,target_class,q25,q50,q75
0,human,0.008706,0.048100,0.176889
1,animal,0.002385,0.008889,0.030272
2,vehicle,0.008718,0.044408,0.193054


In [9]:
bbox_group_counts = (
    eligible_candidates
    .groupby(
        [
            "target_class",
            "relative_bbox_group"
        ],
        observed=True
    )
    .size()
    .rename("eligible_candidates")
    .reset_index()
)

bbox_group_counts[
    "relative_bbox_group"
] = pd.Categorical(
    bbox_group_counts[
        "relative_bbox_group"
    ],
    categories=BBOX_GROUP_LABELS,
    ordered=True
)

bbox_group_counts = (
    bbox_group_counts
    .sort_values(
        [
            "target_class",
            "relative_bbox_group"
        ]
    )
    .reset_index(drop=True)
)

display(bbox_group_counts)

print(
    "Missing bbox groups:",
    eligible_candidates[
        "relative_bbox_group"
    ].isna().sum()
)

,target_class,relative_bbox_group,eligible_candidates
0,animal,very_small,280
1,animal,small,279
2,animal,medium,279
3,animal,large,280
4,human,very_small,280
5,human,small,279
6,human,medium,279
7,human,large,279
8,vehicle,very_small,279
9,vehicle,small,279


Missing bbox groups: 0


In [10]:
bbox_group_thresholds.to_csv(
    BBOX_THRESHOLDS_PATH,
    index=False
)

print("Saved:", BBOX_THRESHOLDS_PATH)

Saved: sampling_outputs\relative_bbox_group_thresholds.csv


## Final positive-selection feasibility

Before final selection, each class × confidence-band × time-window cell is
checked for sufficient eligible records, unique sites and representation across
the balancing variables.

The final design requires 20 records per cell and allows at most one selected
record per site within each target class.

In [11]:
cell_feasibility = (
    eligible_candidates
    .groupby(
        CELL_COLUMNS,
        observed=True
    )
    .agg(
        eligible_candidates=(
            "candidate_record_id",
            "size"
        ),
        unique_images=(
            "photo_id",
            "nunique"
        ),
        unique_sequences=(
            "sequence_id",
            "nunique"
        ),
        unique_sites=(
            "site_id",
            "nunique"
        ),
        bbox_groups=(
            "relative_bbox_group",
            "nunique"
        ),
        geographic_groups=(
            "geographic_group",
            "nunique"
        ),
        seasons=(
            "season",
            "nunique"
        )
    )
    .reset_index()
)

cell_feasibility[
    "locally_feasible"
] = (
    cell_feasibility[
        "eligible_candidates"
    ].ge(FINAL_PER_CELL)
    & cell_feasibility[
        "unique_sites"
    ].ge(FINAL_PER_CELL)
)

print("Sampling cells:", len(cell_feasibility))

print(
    "Minimum eligible candidates:",
    int(
        cell_feasibility[
            "eligible_candidates"
        ].min()
    )
)

print(
    "Minimum unique sites:",
    int(
        cell_feasibility[
            "unique_sites"
        ].min()
    )
)

print(
    "Minimum bbox groups represented:",
    int(
        cell_feasibility[
            "bbox_groups"
        ].min()
    )
)

print(
    "Minimum geographic groups represented:",
    int(
        cell_feasibility[
            "geographic_groups"
        ].min()
    )
)

print(
    "Minimum seasons represented:",
    int(
        cell_feasibility[
            "seasons"
        ].min()
    )
)

print(
    "Locally infeasible cells:",
    int(
        (~cell_feasibility[
            "locally_feasible"
        ]).sum()
    )
)

Sampling cells: 84
Minimum eligible candidates: 39
Minimum unique sites: 29
Minimum bbox groups represented: 3
Minimum geographic groups represented: 4
Minimum seasons represented: 3
Locally infeasible cells: 0


In [12]:
most_constrained_cells = (
    cell_feasibility
    .sort_values(
        [
            "unique_sites",
            "eligible_candidates",
            "geographic_groups",
            "seasons",
            "bbox_groups"
        ],
        ascending=True
    )
    .head(15)
    .reset_index(drop=True)
)

print(most_constrained_cells)

   target_class sampling_confidence_band sampling_time_window  \
0         human             0.90_to_1.00            overnight   
1       vehicle             0.90_to_1.00   morning_transition   
2       vehicle             0.90_to_1.00            overnight   
3       vehicle             0.70_to_0.90            overnight   
4       vehicle             0.30_to_0.70            overnight   
5         human             0.90_to_1.00   morning_transition   
6         human             0.90_to_1.00         daytime_core   
7         human             0.10_to_0.30         daytime_core   
8         human             0.70_to_0.90            overnight   
9       vehicle             0.70_to_0.90   morning_transition   
10      vehicle             0.90_to_1.00   evening_transition   
11        human             0.05_to_0.10            overnight   
12        human             0.05_to_0.10         daytime_core   
13      vehicle             0.30_to_0.70   morning_transition   
14        human          

In [13]:
bbox_cell_availability = (
    eligible_candidates
    .groupby(
        CELL_COLUMNS
        + ["relative_bbox_group"],
        observed=True
    )
    .size()
    .unstack(
        fill_value=0
    )
    .reindex(
        columns=BBOX_GROUP_LABELS,
        fill_value=0
    )
    .reset_index()
)

bbox_cell_availability[
    "minimum_bbox_group_count"
] = (
    bbox_cell_availability[
        BBOX_GROUP_LABELS
    ]
    .min(axis=1)
)

bbox_cell_availability[
    "bbox_groups_present"
] = (
    bbox_cell_availability[
        BBOX_GROUP_LABELS
    ]
    .gt(0)
    .sum(axis=1)
)

print(
    "Cells containing all four bbox groups:",
    int(
        bbox_cell_availability[
            "bbox_groups_present"
        ].eq(4).sum()
    ),
    "/",
    len(bbox_cell_availability)
)

print(
    "Lowest candidate count in any bbox group within a cell:",
    int(
        bbox_cell_availability[
            "minimum_bbox_group_count"
        ].min()
    )
)

print(
    bbox_cell_availability
    .sort_values(
        [
            "bbox_groups_present",
            "minimum_bbox_group_count"
        ]
    )
    .head(15)
)

Cells containing all four bbox groups: 72 / 84
Lowest candidate count in any bbox group within a cell: 0
relative_bbox_group target_class sampling_confidence_band  \
8                         animal             0.05_to_0.10   
21                        animal             0.70_to_0.90   
23                        animal             0.70_to_0.90   
24                        animal             0.90_to_1.00   
25                        animal             0.90_to_1.00   
26                        animal             0.90_to_1.00   
27                        animal             0.90_to_1.00   
52                         human             0.90_to_1.00   
53                         human             0.90_to_1.00   
55                         human             0.90_to_1.00   
80                       vehicle             0.90_to_1.00   
82                       vehicle             0.90_to_1.00   
4                         animal             0.02_to_0.05   
54                         human         

In [14]:
assert len(cell_feasibility) == 84
assert cell_feasibility[
    "eligible_candidates"
].ge(FINAL_PER_CELL).all()

assert cell_feasibility[
    "unique_sites"
].ge(FINAL_PER_CELL).all()

assert bbox_cell_availability[
    "bbox_groups_present"
].ge(1).all()

print("Local final-selection feasibility passed")

Local final-selection feasibility passed


## Class-level site feasibility

The final positive sample requires 560 records per target class.

Because each site may contribute at most one final record within a target
class, each class must contain at least 560 eligible unique sites across its
28 sampling cells.

Site overlap across cells is also inspected before final selection.

In [15]:
POSITIVE_CELLS_PER_CLASS = 28
FINAL_PER_CLASS = (
    POSITIVE_CELLS_PER_CLASS
    * FINAL_PER_CELL
)

class_cell_counts = (
    eligible_candidates[
        CELL_COLUMNS
    ]
    .drop_duplicates()
    .groupby(
        "target_class",
        observed=True
    )
    .size()
    .rename("sampling_cells")
)

class_global_feasibility = (
    eligible_candidates
    .groupby(
        "target_class",
        observed=True
    )
    .agg(
        eligible_candidates=(
            "candidate_record_id",
            "size"
        ),
        unique_images=(
            "photo_id",
            "nunique"
        ),
        unique_sequences=(
            "sequence_id",
            "nunique"
        ),
        unique_sites=(
            "site_id",
            "nunique"
        )
    )
    .join(class_cell_counts)
    .reset_index()
)

class_global_feasibility[
    "required_final_records"
] = FINAL_PER_CLASS

class_global_feasibility[
    "site_surplus"
] = (
    class_global_feasibility[
        "unique_sites"
    ]
    - FINAL_PER_CLASS
)

class_global_feasibility[
    "class_level_feasible"
] = (
    class_global_feasibility[
        "sampling_cells"
    ].eq(POSITIVE_CELLS_PER_CLASS)
    & class_global_feasibility[
        "unique_sites"
    ].ge(FINAL_PER_CLASS)
)

display(class_global_feasibility)

,target_class,eligible_candidates,unique_images,unique_sequences,unique_sites,sampling_cells,required_final_records,site_surplus,class_level_feasible
0,animal,1118,1118,1118,780,28,560,220,True
1,human,1117,1117,1117,718,28,560,158,True
2,vehicle,1116,1116,1116,678,28,560,118,True


In [16]:
site_cell_overlap = (
    eligible_candidates[
        CELL_COLUMNS
        + ["site_id"]
    ]
    .drop_duplicates()
    .groupby(
        [
            "target_class",
            "site_id"
        ],
        observed=True
    )
    .size()
    .rename("cells_available_in")
    .reset_index()
)

site_overlap_summary = (
    site_cell_overlap
    .groupby(
        "target_class",
        observed=True
    )["cells_available_in"]
    .agg(
        unique_sites="size",
        median_cells_per_site="median",
        maximum_cells_per_site="max"
    )
    .reset_index()
)

sites_in_multiple_cells = (
    site_cell_overlap.loc[
        site_cell_overlap[
            "cells_available_in"
        ].gt(1)
    ]
    .groupby(
        "target_class",
        observed=True
    )
    .size()
    .reindex(
        TARGET_CLASSES,
        fill_value=0
    )
    .rename("sites_in_multiple_cells")
    .reset_index()
)

site_overlap_summary = (
    site_overlap_summary
    .merge(
        sites_in_multiple_cells,
        on="target_class",
        how="left"
    )
)

display(site_overlap_summary)

,target_class,unique_sites,median_cells_per_site,maximum_cells_per_site,sites_in_multiple_cells
0,animal,780,1.0,2,310
1,human,718,1.0,2,277
2,vehicle,678,1.0,2,322


In [17]:
assert class_global_feasibility[
    "class_level_feasible"
].all()

print(
    "Required final records per class:",
    FINAL_PER_CLASS
)

print("Class-level site feasibility passed")

Required final records per class: 560
Class-level site feasibility passed


## Balanced final positive selection

Twenty records are selected from each class × confidence-band × time-window
cell.

Cells with fewer available sites are processed first. Within each cell,
selection favours sites that occur in fewer remaining cells, reducing the risk
that a site needed by another cell is used prematurely.

Among feasible candidates, relative bounding-box group, geographic group and
season are balanced using a reproducible greedy procedure.

A site may contribute at most one final record within each target class.

In [18]:
BALANCE_COLUMNS = [
    "relative_bbox_group",
    "geographic_group",
    "season"
]


def select_positive_class(
    class_candidates,
    class_feasibility,
    final_per_cell,
    random_seed,
    max_attempts=100
):
    class_candidates = class_candidates.copy()

    class_candidates["_cell_key"] = list(
        zip(
            class_candidates[
                "sampling_confidence_band"
            ],
            class_candidates[
                "sampling_time_window"
            ]
        )
    )

    constraint_columns = [
        "unique_sites",
        "eligible_candidates",
        "bbox_groups",
        "geographic_groups",
        "seasons"
    ]

    for attempt in range(max_attempts):
        rng = np.random.default_rng(
            random_seed + attempt
        )

        ordered_cells = class_feasibility.copy()

        ordered_cells["_random_tie"] = rng.random(
            len(ordered_cells)
        )

        ordered_cells = (
            ordered_cells
            .sort_values(
                constraint_columns
                + ["_random_tie"],
                ascending=True
            )
            .reset_index(drop=True)
        )

        cell_keys = list(
            zip(
                ordered_cells[
                    "sampling_confidence_band"
                ],
                ordered_cells[
                    "sampling_time_window"
                ]
            )
        )

        used_sites = set()
        selected_parts = []
        diagnostic_rows = []
        attempt_failed = False

        for cell_position in range(
            len(ordered_cells)
        ):
            cell_key = cell_keys[
                cell_position
            ]

            confidence_band = cell_key[0]
            time_window = cell_key[1]

            available_pool = (
                class_candidates.loc[
                    class_candidates[
                        "sampling_confidence_band"
                    ].eq(confidence_band)
                    & class_candidates[
                        "sampling_time_window"
                    ].eq(time_window)
                    & ~class_candidates[
                        "site_id"
                    ].isin(used_sites)
                ]
                .copy()
            )

            available_site_count = (
                available_pool[
                    "site_id"
                ].nunique()
            )

            if (
                available_site_count
                < final_per_cell
            ):
                attempt_failed = True
                break

            remaining_cell_keys = list(
                cell_keys[cell_position:]
            )

            remaining_candidates = (
                class_candidates.loc[
                    class_candidates[
                        "_cell_key"
                    ].isin(
                        remaining_cell_keys
                    )
                    & ~class_candidates[
                        "site_id"
                    ].isin(used_sites)
                ]
                .copy()
            )

            site_cell_options = (
                remaining_candidates[
                    [
                        "site_id",
                        "_cell_key"
                    ]
                ]
                .drop_duplicates()
                .groupby(
                    "site_id",
                    observed=True
                )
                .size()
            )

            available_pool[
                "_remaining_cell_options"
            ] = (
                available_pool[
                    "site_id"
                ]
                .map(site_cell_options)
                .fillna(1)
            )

            selected_indices = []

            balance_counts = {
                column: {}
                for column in BALANCE_COLUMNS
            }

            current_pool = (
                available_pool.copy()
            )

            for selection_rank in range(
                1,
                final_per_cell + 1
            ):
                if current_pool.empty:
                    attempt_failed = True
                    break

                current_pool[
                    "_balance_score"
                ] = 0

                for column in BALANCE_COLUMNS:
                    current_pool[
                        "_balance_score"
                    ] += (
                        current_pool[column]
                        .map(
                            balance_counts[
                                column
                            ]
                        )
                        .fillna(0)
                    )

                current_pool[
                    "_random_tie"
                ] = rng.random(
                    len(current_pool)
                )

                selected_index = (
                    current_pool
                    .sort_values(
                        [
                            "_remaining_cell_options",
                            "_balance_score",
                            "_random_tie"
                        ],
                        ascending=True
                    )
                    .index[0]
                )

                selected_row = (
                    current_pool.loc[
                        selected_index
                    ]
                )

                selected_indices.append(
                    selected_index
                )

                selected_site = (
                    selected_row[
                        "site_id"
                    ]
                )

                for column in BALANCE_COLUMNS:
                    selected_value = (
                        selected_row[column]
                    )

                    balance_counts[
                        column
                    ][selected_value] = (
                        balance_counts[
                            column
                        ].get(
                            selected_value,
                            0
                        )
                        + 1
                    )

                current_pool = (
                    current_pool.loc[
                        ~current_pool[
                            "site_id"
                        ].eq(selected_site)
                    ]
                    .copy()
                )

            if attempt_failed:
                break

            selected_cell = (
                class_candidates.loc[
                    selected_indices
                ]
                .copy()
            )

            selected_cell[
                "selection_rank_within_cell"
            ] = range(
                1,
                final_per_cell + 1
            )

            selected_parts.append(
                selected_cell
            )

            used_sites.update(
                selected_cell[
                    "site_id"
                ].tolist()
            )

            diagnostic_rows.append({
                "target_class": (
                    selected_cell[
                        "target_class"
                    ].iloc[0]
                ),
                "sampling_confidence_band": (
                    confidence_band
                ),
                "sampling_time_window": (
                    time_window
                ),
                "available_records_before_selection": (
                    len(available_pool)
                ),
                "available_sites_before_selection": (
                    available_site_count
                ),
                "selected_records": (
                    len(selected_cell)
                ),
                "selected_bbox_groups": (
                    selected_cell[
                        "relative_bbox_group"
                    ].nunique()
                ),
                "selected_geographic_groups": (
                    selected_cell[
                        "geographic_group"
                    ].nunique()
                ),
                "selected_seasons": (
                    selected_cell[
                        "season"
                    ].nunique()
                ),
                "selection_attempt": (
                    attempt + 1
                )
            })

        if (
            not attempt_failed
            and len(selected_parts)
            == len(ordered_cells)
        ):
            selected_class = pd.concat(
                selected_parts,
                ignore_index=True
            )

            class_diagnostics = (
                pd.DataFrame(
                    diagnostic_rows
                )
            )

            return (
                selected_class,
                class_diagnostics
            )

    raise RuntimeError(
        "Final selection failed after "
        f"{max_attempts} attempts."
    )

In [19]:
selected_class_parts = []
selection_diagnostic_parts = []

for class_number, target_class in enumerate(
    TARGET_CLASSES
):
    class_candidates = (
        eligible_candidates.loc[
            eligible_candidates[
                "target_class"
            ].eq(target_class)
        ]
        .copy()
    )

    class_feasibility = (
        cell_feasibility.loc[
            cell_feasibility[
                "target_class"
            ].eq(target_class)
        ]
        .copy()
    )

    selected_class, class_diagnostics = (
        select_positive_class(
            class_candidates=class_candidates,
            class_feasibility=class_feasibility,
            final_per_cell=FINAL_PER_CELL,
            random_seed=(
                RANDOM_SEED
                + class_number * 10_000
            )
        )
    )

    selected_class_parts.append(
        selected_class
    )

    selection_diagnostic_parts.append(
        class_diagnostics
    )

    print(
        "Selected",
        target_class,
        "records:",
        len(selected_class)
    )

final_positive_records = pd.concat(
    selected_class_parts,
    ignore_index=True
)

positive_selection_diagnostics = pd.concat(
    selection_diagnostic_parts,
    ignore_index=True
)

final_positive_records = (
    final_positive_records
    .drop(
        columns=[
            "_cell_key"
        ],
        errors="ignore"
    )
    .reset_index(drop=True)
)

final_positive_records[
    "final_positive_record_id"
] = [
    f"FINAL_POS_{record_number:04d}"
    for record_number in range(
        1,
        len(final_positive_records) + 1
    )
]

Selected human records: 560
Selected animal records: 560
Selected vehicle records: 560


## Final positive-sample validation

The selected positive sample is checked for exact cell sizes, class totals,
site and sequence uniqueness, valid image access and valid relative
bounding-box areas.

In [20]:
final_cell_validation = (
    final_positive_records
    .groupby(
        CELL_COLUMNS,
        observed=True
    )
    .agg(
        selected_records=(
            "candidate_record_id",
            "size"
        ),
        unique_images=(
            "photo_id",
            "nunique"
        ),
        unique_sequences=(
            "sequence_id",
            "nunique"
        ),
        unique_sites=(
            "site_id",
            "nunique"
        )
    )
    .reset_index()
)

final_class_validation = (
    final_positive_records
    .groupby(
        "target_class",
        observed=True
    )
    .agg(
        selected_records=(
            "candidate_record_id",
            "size"
        ),
        unique_images=(
            "photo_id",
            "nunique"
        ),
        unique_sequences=(
            "sequence_id",
            "nunique"
        ),
        unique_sites=(
            "site_id",
            "nunique"
        )
    )
    .reindex(TARGET_CLASSES)
    .reset_index()
)

display(final_class_validation)

print(
    "Final positive class-records:",
    len(final_positive_records)
)

print(
    "Completed sampling cells:",
    len(final_cell_validation)
)

print(
    "Minimum records per cell:",
    int(
        final_cell_validation[
            "selected_records"
        ].min()
    )
)

print(
    "Maximum records per cell:",
    int(
        final_cell_validation[
            "selected_records"
        ].max()
    )
)

print(
    "Duplicated candidate record IDs:",
    int(
        final_positive_records[
            "candidate_record_id"
        ].duplicated().sum()
    )
)

print(
    "Duplicated final record IDs:",
    int(
        final_positive_records[
            "final_positive_record_id"
        ].duplicated().sum()
    )
)

print(
    "Duplicated sites within class:",
    int(
        final_positive_records.duplicated(
            subset=[
                "target_class",
                "site_id"
            ]
        ).sum()
    )
)

print(
    "Duplicated sequences within class:",
    int(
        final_positive_records.duplicated(
            subset=[
                "target_class",
                "sequence_id"
            ]
        ).sum()
    )
)

,target_class,selected_records,unique_images,unique_sequences,unique_sites
0,human,560,560,560,560
1,animal,560,560,560,560
2,vehicle,560,560,560,560


Final positive class-records: 1680
Completed sampling cells: 84
Minimum records per cell: 20
Maximum records per cell: 20
Duplicated candidate record IDs: 0
Duplicated final record IDs: 0
Duplicated sites within class: 0
Duplicated sequences within class: 0


## Positive-sample diversity diagnostics

The final positive sample is compared with the eligible candidate pool across
relative bounding-box group, geographic group and season.

These variables were used as balancing variables rather than fixed sampling
strata. Exact equality is therefore not required, particularly where some
sampling cells do not contain every category.

In [21]:
selected_cell_diversity = (
    final_positive_records
    .groupby(
        CELL_COLUMNS,
        observed=True
    )
    .agg(
        selected_bbox_groups=(
            "relative_bbox_group",
            "nunique"
        ),
        selected_geographic_groups=(
            "geographic_group",
            "nunique"
        ),
        selected_seasons=(
            "season",
            "nunique"
        )
    )
    .reset_index()
)

diversity_summary = (
    selected_cell_diversity[
        [
            "selected_bbox_groups",
            "selected_geographic_groups",
            "selected_seasons"
        ]
    ]
    .agg(
        [
            "min",
            "median",
            "max"
        ]
    )
    .T
    .reset_index()
    .rename(
        columns={
            "index": "diversity_variable"
        }
    )
)

display(diversity_summary)

,diversity_variable,min,median,max
0,selected_bbox_groups,2.0,4.0,4.0
1,selected_geographic_groups,4.0,5.0,5.0
2,selected_seasons,3.0,4.0,4.0


In [22]:
least_diverse_cells = (
    selected_cell_diversity
    .sort_values(
        [
            "selected_bbox_groups",
            "selected_geographic_groups",
            "selected_seasons"
        ]
    )
    .head(15)
    .reset_index(drop=True)
)

display(least_diverse_cells)

,target_class,sampling_confidence_band,sampling_time_window,selected_bbox_groups,selected_geographic_groups,selected_seasons
0,vehicle,0.90_to_1.00,daytime_core,2,4,4
1,animal,0.90_to_1.00,morning_transition,2,5,4
2,animal,0.90_to_1.00,overnight,2,5,4
3,human,0.02_to_0.05,evening_transition,3,4,4
4,human,0.90_to_1.00,daytime_core,3,4,4
5,human,0.90_to_1.00,overnight,3,4,4
6,vehicle,0.90_to_1.00,morning_transition,3,4,4
7,animal,0.05_to_0.10,daytime_core,3,5,4
8,animal,0.70_to_0.90,evening_transition,3,5,4
9,animal,0.70_to_0.90,morning_transition,3,5,4


In [23]:
selected_bbox_counts = (
    final_positive_records
    .groupby(
        [
            "target_class",
            "relative_bbox_group"
        ],
        observed=True
    )
    .size()
    .unstack(
        fill_value=0
    )
    .reindex(
        columns=BBOX_GROUP_LABELS,
        fill_value=0
    )
    .reindex(TARGET_CLASSES)
)

selected_bbox_percentages = (
    selected_bbox_counts
    .div(
        selected_bbox_counts.sum(axis=1),
        axis=0
    )
    .mul(100)
    .round(1)
)

print("Selected bbox counts")
display(selected_bbox_counts)

print("Selected bbox percentages")
display(selected_bbox_percentages)

Selected bbox counts


relative_bbox_group,very_small,small,medium,large
target_class,,,,
human,128,149,135,148
animal,124,141,151,144
vehicle,126,143,142,149


Selected bbox percentages


relative_bbox_group,very_small,small,medium,large
target_class,,,,
human,22.9,26.6,24.1,26.4
animal,22.1,25.2,27.0,25.7
vehicle,22.5,25.5,25.4,26.6


In [24]:
selected_geographic_counts = (
    final_positive_records
    .groupby(
        [
            "target_class",
            "geographic_group"
        ],
        observed=True
    )
    .size()
    .unstack(
        fill_value=0
    )
    .reindex(TARGET_CLASSES)
)

selected_season_counts = (
    final_positive_records
    .groupby(
        [
            "target_class",
            "season"
        ],
        observed=True
    )
    .size()
    .unstack(
        fill_value=0
    )
    .reindex(TARGET_CLASSES)
)

print("Selected geographic-group counts")
display(selected_geographic_counts)

print("Selected season counts")
display(selected_season_counts)

Selected geographic-group counts


geographic_group,north,north_central,outside_uk_core,south,south_central
target_class,,,,,
human,146,136,24,120,134
animal,121,122,82,118,117
vehicle,140,131,13,141,135


Selected season counts


season,autumn,spring,summer,winter
target_class,,,,
human,160,136,155,109
animal,152,133,149,126
vehicle,156,150,164,90


In [25]:
def compare_candidate_and_selected_distribution(
    candidate_data,
    selected_data,
    variable
):
    candidate_distribution = (
        candidate_data
        .groupby(
            [
                "target_class",
                variable
            ],
            observed=True
        )
        .size()
        .rename("eligible_count")
        .reset_index()
    )

    selected_distribution = (
        selected_data
        .groupby(
            [
                "target_class",
                variable
            ],
            observed=True
        )
        .size()
        .rename("selected_count")
        .reset_index()
    )

    comparison = (
        candidate_distribution
        .merge(
            selected_distribution,
            on=[
                "target_class",
                variable
            ],
            how="outer"
        )
        .fillna(0)
    )

    comparison[
        "eligible_percentage"
    ] = (
        comparison[
            "eligible_count"
        ]
        / comparison
        .groupby(
            "target_class"
        )["eligible_count"]
        .transform("sum")
        * 100
    )

    comparison[
        "selected_percentage"
    ] = (
        comparison[
            "selected_count"
        ]
        / comparison
        .groupby(
            "target_class"
        )["selected_count"]
        .transform("sum")
        * 100
    )

    comparison[
        "percentage_point_difference"
    ] = (
        comparison[
            "selected_percentage"
        ]
        - comparison[
            "eligible_percentage"
        ]
    )

    percentage_columns = [
        "eligible_percentage",
        "selected_percentage",
        "percentage_point_difference"
    ]

    comparison[
        percentage_columns
    ] = comparison[
        percentage_columns
    ].round(1)

    return comparison.sort_values(
        [
            "target_class",
            variable
        ]
    ).reset_index(drop=True)

In [26]:
bbox_distribution_comparison = (
    compare_candidate_and_selected_distribution(
        eligible_candidates,
        final_positive_records,
        "relative_bbox_group"
    )
)

geographic_distribution_comparison = (
    compare_candidate_and_selected_distribution(
        eligible_candidates,
        final_positive_records,
        "geographic_group"
    )
)

season_distribution_comparison = (
    compare_candidate_and_selected_distribution(
        eligible_candidates,
        final_positive_records,
        "season"
    )
)

print("BBox distribution comparison")
display(bbox_distribution_comparison)

print("Geographic distribution comparison")
display(geographic_distribution_comparison)

print("Season distribution comparison")
display(season_distribution_comparison)

BBox distribution comparison


,target_class,relative_bbox_group,eligible_count,selected_count,eligible_percentage,selected_percentage,percentage_point_difference
0,animal,large,280,144,25.0,25.7,0.7
1,animal,medium,279,151,25.0,27.0,2.0
2,animal,small,279,141,25.0,25.2,0.2
3,animal,very_small,280,124,25.0,22.1,-2.9
4,human,large,279,148,25.0,26.4,1.5
5,human,medium,279,135,25.0,24.1,-0.9
6,human,small,279,149,25.0,26.6,1.6
7,human,very_small,280,128,25.1,22.9,-2.2
8,vehicle,large,279,149,25.0,26.6,1.6
9,vehicle,medium,279,142,25.0,25.4,0.4


Geographic distribution comparison


,target_class,geographic_group,eligible_count,selected_count,eligible_percentage,selected_percentage,percentage_point_difference
0,animal,north,228,121,20.4,21.6,1.2
1,animal,north_central,232,122,20.8,21.8,1.0
2,animal,outside_uk_core,194,82,17.4,14.6,-2.7
3,animal,south,228,118,20.4,21.1,0.7
4,animal,south_central,236,117,21.1,20.9,-0.2
5,human,north,298,146,26.7,26.1,-0.6
6,human,north_central,274,136,24.5,24.3,-0.2
7,human,outside_uk_core,41,24,3.7,4.3,0.6
8,human,south,238,120,21.3,21.4,0.1
9,human,south_central,266,134,23.8,23.9,0.1


Season distribution comparison


,target_class,season,eligible_count,selected_count,eligible_percentage,selected_percentage,percentage_point_difference
0,animal,autumn,283,152,25.3,27.1,1.8
1,animal,spring,280,133,25.0,23.8,-1.3
2,animal,summer,269,149,24.1,26.6,2.5
3,animal,winter,286,126,25.6,22.5,-3.1
4,human,autumn,320,160,28.6,28.6,-0.1
5,human,spring,281,136,25.2,24.3,-0.9
6,human,summer,310,155,27.8,27.7,-0.1
7,human,winter,206,109,18.4,19.5,1.0
8,vehicle,autumn,317,156,28.4,27.9,-0.5
9,vehicle,spring,306,150,27.4,26.8,-0.6


In [27]:
bbox_diversity_check = (
    selected_cell_diversity
    .merge(
        cell_feasibility[
            CELL_COLUMNS
            + [
                "bbox_groups",
                "geographic_groups",
                "seasons"
            ]
        ],
        on=CELL_COLUMNS,
        how="left",
        validate="one_to_one"
    )
)

bbox_diversity_check[
    "bbox_groups_lost"
] = (
    bbox_diversity_check[
        "bbox_groups"
    ]
    - bbox_diversity_check[
        "selected_bbox_groups"
    ]
)

low_bbox_diversity_cells = (
    bbox_diversity_check.loc[
        bbox_diversity_check[
            "selected_bbox_groups"
        ].le(2)
    ]
    .sort_values(
        [
            "selected_bbox_groups",
            "bbox_groups_lost"
        ],
        ascending=[
            True,
            False
        ]
    )
    .reset_index(drop=True)
)

print(low_bbox_diversity_cells)

print(
    "Cells with only two selected bbox groups:",
    len(low_bbox_diversity_cells)
)

  target_class sampling_confidence_band sampling_time_window  \
0       animal             0.90_to_1.00   morning_transition   
1       animal             0.90_to_1.00            overnight   
2      vehicle             0.90_to_1.00         daytime_core   

   selected_bbox_groups  selected_geographic_groups  selected_seasons  \
0                     2                           5                 4   
1                     2                           5                 4   
2                     2                           4                 4   

   bbox_groups  geographic_groups  seasons  bbox_groups_lost  
0            3                  5        4                 1  
1            3                  5        4                 1  
2            3                  4        4                 1  
Cells with only two selected bbox groups: 3


In [28]:
low_bbox_cell_columns = (
    low_bbox_diversity_cells[
        CELL_COLUMNS
    ]
)

low_bbox_candidate_counts = (
    eligible_candidates
    .merge(
        low_bbox_cell_columns,
        on=CELL_COLUMNS,
        how="inner"
    )
    .groupby(
        CELL_COLUMNS
        + ["relative_bbox_group"],
        observed=True
    )
    .size()
    .rename("eligible_count")
    .reset_index()
)

low_bbox_selected_counts = (
    final_positive_records
    .merge(
        low_bbox_cell_columns,
        on=CELL_COLUMNS,
        how="inner"
    )
    .groupby(
        CELL_COLUMNS
        + ["relative_bbox_group"],
        observed=True
    )
    .size()
    .rename("selected_count")
    .reset_index()
)

low_bbox_group_comparison = (
    low_bbox_candidate_counts
    .merge(
        low_bbox_selected_counts,
        on=(
            CELL_COLUMNS
            + ["relative_bbox_group"]
        ),
        how="left"
    )
    .fillna({
        "selected_count": 0
    })
)

display(low_bbox_group_comparison)

,target_class,sampling_confidence_band,sampling_time_window,relative_bbox_group,eligible_count,selected_count
0,animal,0.90_to_1.00,morning_transition,large,31,15.0
1,animal,0.90_to_1.00,morning_transition,medium,7,5.0
2,animal,0.90_to_1.00,morning_transition,small,2,0.0
3,animal,0.90_to_1.00,overnight,large,29,11.0
4,animal,0.90_to_1.00,overnight,medium,10,9.0
5,animal,0.90_to_1.00,overnight,small,1,0.0
6,vehicle,0.90_to_1.00,daytime_core,large,26,13.0
7,vehicle,0.90_to_1.00,daytime_core,medium,13,7.0
8,vehicle,0.90_to_1.00,daytime_core,small,1,0.0


## Save final positive sample

The validated positive class-record sample and selection diagnostics are saved
as reproducible outputs.

The final sample contains 20 records in each of the 84 positive sampling cells,
with one selected record per site and sequence within each target class.

In [29]:
POSITIVE_DIAGNOSTICS_PATH = (
    OUTPUT_DIR
    / "final_positive_sampling_diagnostics.csv"
)

In [30]:
final_positive_records[
    "sampling_cell_id"
] = (
    final_positive_records[
        "target_class"
    ].astype(str)
    + "__"
    + final_positive_records[
        "sampling_confidence_band"
    ].astype(str)
    + "__"
    + final_positive_records[
        "sampling_time_window"
    ].astype(str)
)

In [31]:
final_positive_records.to_csv(
    FINAL_POSITIVE_RECORDS_PATH,
    index=False
)

positive_selection_diagnostics.to_csv(
    POSITIVE_DIAGNOSTICS_PATH,
    index=False
)

print("Saved:", FINAL_POSITIVE_RECORDS_PATH)
print("Saved:", POSITIVE_DIAGNOSTICS_PATH)

Saved: sampling_outputs\final_positive_class_records.csv
Saved: sampling_outputs\final_positive_sampling_diagnostics.csv


## Deduplicated positive image queue

The final positive sample is stored at class-record level because one image may
contribute to more than one target-class evaluation.

For manual annotation, selected records are collapsed to one row per physical
photo. The image-level queue retains indicators showing whether each image was
selected for human, animal or vehicle evaluation.

In [32]:
COMMON_IMAGE_COLUMN_CANDIDATES = [
    "sequence_id",
    "sequence_num",
    "site_id",
    "person_id",
    "filename",
    "dirname",
    "image_url",
    "image_width_px",
    "image_height_px",
    "image_format",
    "image_access_status",
    "sampling_time_window",
    "season",
    "geographic_group"
]

common_image_columns = [
    column
    for column in COMMON_IMAGE_COLUMN_CANDIDATES
    if column in final_positive_records.columns
]

metadata_conflicts = []

for column in common_image_columns:
    maximum_values_per_photo = (
        final_positive_records
        .groupby(
            "photo_id",
            observed=True
        )[column]
        .nunique(dropna=False)
        .max()
    )

    if maximum_values_per_photo > 1:
        metadata_conflicts.append(column)

print(
    "Common image columns:",
    common_image_columns
)

print(
    "Metadata conflicts:",
    metadata_conflicts
)

assert not metadata_conflicts

Common image columns: ['sequence_id', 'sequence_num', 'site_id', 'filename', 'dirname', 'image_url', 'image_width_px', 'image_height_px', 'image_format', 'image_access_status', 'sampling_time_window', 'season', 'geographic_group']
Metadata conflicts: []


In [33]:
def join_unique_values(series):
    values = sorted({
        str(value)
        for value in series.dropna()
    })

    return "|".join(values)


def join_target_classes(series):
    present_classes = set(
        series.dropna().astype(str)
    )

    ordered_classes = [
        target_class
        for target_class in TARGET_CLASSES
        if target_class in present_classes
    ]

    return "|".join(ordered_classes)

In [34]:
image_aggregation = {
    column: (
        column,
        "first"
    )
    for column in common_image_columns
}

image_aggregation.update({
    "positive_target_classes": (
        "target_class",
        join_target_classes
    ),
    "positive_candidate_record_ids": (
        "candidate_record_id",
        join_unique_values
    ),
    "positive_final_record_ids": (
        "final_positive_record_id",
        join_unique_values
    ),
    "positive_sampling_cell_ids": (
        "sampling_cell_id",
        join_unique_values
    ),
    "number_positive_target_classes": (
        "target_class",
        "nunique"
    )
})

final_positive_image_queue = (
    final_positive_records
    .sort_values(
        "final_positive_record_id"
    )
    .groupby(
        "photo_id",
        as_index=False,
        observed=True
    )
    .agg(**image_aggregation)
)

In [35]:
for target_class in TARGET_CLASSES:
    selected_photo_ids = set(
        final_positive_records.loc[
            final_positive_records[
                "target_class"
            ].eq(target_class),
            "photo_id"
        ]
    )

    final_positive_image_queue[
        f"selected_for_{target_class}"
    ] = (
        final_positive_image_queue[
            "photo_id"
        ].isin(selected_photo_ids)
    )

final_positive_image_queue[
    "positive_image_queue_id"
] = [
    f"POS_IMG_{image_number:04d}"
    for image_number in range(
        1,
        len(final_positive_image_queue) + 1
    )
]

In [36]:
selection_flag_columns = [
    f"selected_for_{target_class}"
    for target_class in TARGET_CLASSES
]

assert final_positive_image_queue[
    "photo_id"
].is_unique

assert set(
    final_positive_image_queue[
        "photo_id"
    ]
) == set(
    final_positive_records[
        "photo_id"
    ]
)

assert (
    final_positive_image_queue[
        selection_flag_columns
    ]
    .sum(axis=1)
    .eq(
        final_positive_image_queue[
            "number_positive_target_classes"
        ]
    )
    .all()
)

assert final_positive_image_queue[
    "number_positive_target_classes"
].between(1, 3).all()

target_count_summary = (
    final_positive_image_queue[
        "number_positive_target_classes"
    ]
    .value_counts()
    .sort_index()
    .rename_axis(
        "number_positive_target_classes"
    )
    .reset_index(
        name="unique_images"
    )
)

print(
    "Positive class-records:",
    f"{len(final_positive_records):,}"
)

print(
    "Unique positive images:",
    f"{len(final_positive_image_queue):,}"
)

print(
    "Repeated annotation avoided:",
    f"{(
        len(final_positive_records)
        - len(final_positive_image_queue)
    ):,}"
)

display(target_count_summary)

Positive class-records: 1,680
Unique positive images: 1,677
Repeated annotation avoided: 3


,number_positive_target_classes,unique_images
0,1,1674
1,2,3


In [37]:
final_positive_image_queue.to_csv(
    FINAL_POSITIVE_QUEUE_PATH,
    index=False
)

print(
    "Saved:",
    FINAL_POSITIVE_QUEUE_PATH
)

print("Final positive image queue saved")

Saved: sampling_outputs\final_positive_image_queue.csv
Final positive image queue saved


## Zero-confidence pool preparation

A separate zero-confidence sample will be added to represent images without a
positive MegaDetector score.

Before defining this pool, the prepared dataset is inspected to distinguish
explicit score values of zero from missing confidence values. Missing values
must not automatically be treated as genuine zero-confidence predictions.

In [38]:
SAMPLING_PREPARED_PATH = Path(
    "sampling_prepared.csv"
)

FINAL_ZERO_SAMPLE_PATH = (
    OUTPUT_DIR
    / "final_zero_confidence_sample.csv"
)

In [39]:
sampling_prepared = pd.read_csv(
    SAMPLING_PREPARED_PATH,
    low_memory=False
)

print(
    "Prepared dataset records:",
    f"{len(sampling_prepared):,}"
)

print(
    "Prepared dataset columns:",
    len(sampling_prepared.columns)
)

Prepared dataset records: 6,720,367
Prepared dataset columns: 64


In [40]:
score_column_matches = {}

for target_class in TARGET_CLASSES:
    matches = [
        column
        for column in sampling_prepared.columns
        if target_class in column.lower()
        and any(
            keyword in column.lower()
            for keyword in [
                "confidence",
                "score",
                "prob"
            ]
        )
    ]

    score_column_matches[
        target_class
    ] = matches

    print(
        target_class,
        "score-column matches:",
        matches
    )

human score-column matches: ['mega_human_confidence', 'human_bbox_prob', 'human_confidence_band']
animal score-column matches: ['mega_animal_confidence', 'animal_bbox_prob', 'animal_confidence_band']
vehicle score-column matches: ['mega_vehicle_confidence', 'vehicle_bbox_prob', 'vehicle_confidence_band']


In [41]:
detector_metadata_columns = [
    column
    for column in sampling_prepared.columns
    if any(
        keyword in column.lower()
        for keyword in [
            "mega",
            "origin",
            "status",
            "classify"
        ]
    )
]

print("Potential detector metadata columns:")

print(detector_metadata_columns)

Potential detector metadata columns:
['status', 'mega_human_confidence', 'mega_animal_confidence', 'mega_vehicle_confidence']


In [42]:
matched_score_columns = sorted({
    column
    for matches in score_column_matches.values()
    for column in matches
})

score_summary_rows = []

for column in matched_score_columns:
    numeric_values = pd.to_numeric(
        sampling_prepared[column],
        errors="coerce"
    )

    score_summary_rows.append({
        "column": column,
        "non_missing": (
            numeric_values.notna().sum()
        ),
        "missing": (
            numeric_values.isna().sum()
        ),
        "equal_zero": (
            numeric_values.eq(0).sum()
        ),
        "above_zero": (
            numeric_values.gt(0).sum()
        ),
        "below_zero": (
            numeric_values.lt(0).sum()
        ),
        "minimum": (
            numeric_values.min()
        ),
        "maximum": (
            numeric_values.max()
        )
    })

score_value_summary = pd.DataFrame(
    score_summary_rows
)

display(score_value_summary)

,column,non_missing,missing,equal_zero,above_zero,below_zero,minimum,maximum
0,animal_bbox_prob,5579464,1140903,0,5579464,0,0.0101,0.993
1,animal_confidence_band,0,6720367,0,0,0,NaN,NaN
2,human_bbox_prob,614553,6105814,0,614553,0,0.0101,0.986
3,human_confidence_band,0,6720367,0,0,0,NaN,NaN
4,mega_animal_confidence,6720367,0,1140706,5579661,0,0.0000,0.993
5,mega_human_confidence,6720367,0,6105807,614560,0,0.0000,0.986
6,mega_vehicle_confidence,6720367,0,6040430,679937,0,0.0000,0.987
7,vehicle_bbox_prob,679922,6040445,0,679922,0,0.0101,0.987
8,vehicle_confidence_band,0,6720367,0,0,0,NaN,NaN


In [43]:
MEGA_SCORE_COLUMNS = {
    "human": "mega_human_confidence",
    "animal": "mega_animal_confidence",
    "vehicle": "mega_vehicle_confidence"
}

mega_scores = sampling_prepared[
    list(MEGA_SCORE_COLUMNS.values())
].apply(
    pd.to_numeric,
    errors="coerce"
)

sampling_prepared[
    "all_mega_scores_zero"
] = mega_scores.eq(0).all(axis=1)

sampling_prepared[
    "any_mega_score_positive"
] = mega_scores.gt(0).any(axis=1)

print(
    "All three MegaDetector scores zero:",
    f"{sampling_prepared['all_mega_scores_zero'].sum():,}"
)

print(
    "At least one MegaDetector score positive:",
    f"{sampling_prepared['any_mega_score_positive'].sum():,}"
)

print(
    "Neither condition:",
    f"{(
        ~sampling_prepared['all_mega_scores_zero']
        & ~sampling_prepared['any_mega_score_positive']
    ).sum():,}"
)

All three MegaDetector scores zero: 653,130
At least one MegaDetector score positive: 6,067,237
Neither condition: 0


In [44]:
status_by_mega_output = (
    sampling_prepared
    .groupby(
        [
            "all_mega_scores_zero",
            "status"
        ],
        dropna=False
    )
    .size()
    .rename("records")
    .reset_index()
)

display(status_by_mega_output)

,all_mega_scores_zero,status,records
0,False,1,6067237
1,True,1,653130


## Recover metadata for the all-zero MegaDetector pool

The original supplementary export began from the Classify table. Photos with
no stored human, animal or vehicle detection were therefore absent from that
export and lost their supplementary metadata after the left join.

The separately exported Photo and Site records are merged here to recover the
metadata required for zero-confidence sampling.

In [45]:
ZERO_METADATA_PATH = Path(
    "zero_metadata_combined.csv"
)

ZERO_POOL_ENRICHED_PATH = (
    OUTPUT_DIR
    / "zero_confidence_pool_enriched.csv"
)

zero_metadata = pd.read_csv(
    ZERO_METADATA_PATH,
    low_memory=False
)

zero_metadata.columns = (
    zero_metadata.columns
    .str.strip()
)

print(
    "Recovered metadata records:",
    f"{len(zero_metadata):,}"
)

print(
    "Recovered metadata columns:",
    len(zero_metadata.columns)
)

print(
    "Unique recovered photo IDs:",
    f"{zero_metadata['photo_id'].nunique():,}"
)

print(
    "Recovered photo ID range:",
    zero_metadata["photo_id"].min(),
    "to",
    zero_metadata["photo_id"].max()
)

print(
    "Recovered columns:",
    zero_metadata.columns.tolist()
)

Recovered metadata records: 3,559,125
Recovered metadata columns: 12
Unique recovered photo IDs: 3,559,125
Recovered photo ID range: 2 to 26595930
Recovered columns: ['photo_id', 'sequence_id', 'sequence_num', 'site_id', 'person_id', 'taken', 'filename', 'dirname', 'site_name', 'latitude', 'longitude', 'source_chunk']


In [46]:
zero_records = (
    sampling_prepared.loc[
        sampling_prepared[
            "all_mega_scores_zero"
        ]
    ]
    .copy()
)

zero_records["photo_id"] = pd.to_numeric(
    zero_records["photo_id"],
    errors="raise"
)

zero_metadata["photo_id"] = pd.to_numeric(
    zero_metadata["photo_id"],
    errors="raise"
)

print(
    "All-zero records:",
    f"{len(zero_records):,}"
)

print(
    "Unique all-zero photo IDs:",
    f"{zero_records['photo_id'].nunique():,}"
)

print(
    "Duplicated all-zero photo IDs:",
    f"{zero_records['photo_id'].duplicated().sum():,}"
)

print(
    "Duplicated recovered photo IDs:",
    f"{zero_metadata['photo_id'].duplicated().sum():,}"
)

All-zero records: 653,130
Unique all-zero photo IDs: 653,130
Duplicated all-zero photo IDs: 0
Duplicated recovered photo IDs: 0


In [47]:
zero_record_ids = set(
    zero_records["photo_id"]
)

recovered_metadata_ids = set(
    zero_metadata["photo_id"]
)

matched_ids = (
    zero_record_ids
    & recovered_metadata_ids
)

missing_zero_ids = (
    zero_record_ids
    - recovered_metadata_ids
)

extra_metadata_ids = (
    recovered_metadata_ids
    - zero_record_ids
)

print(
    "All-zero photos with recovered metadata:",
    f"{len(matched_ids):,}"
)

print(
    "All-zero photos still unmatched:",
    f"{len(missing_zero_ids):,}"
)

print(
    "Exported metadata photos outside zero pool:",
    f"{len(extra_metadata_ids):,}"
)

All-zero photos with recovered metadata: 652,957
All-zero photos still unmatched: 173
Exported metadata photos outside zero pool: 2,906,168


In [48]:
if missing_zero_ids:
    missing_zero_id_examples = sorted(
        missing_zero_ids
    )[:20]

    print(
        "Example unmatched all-zero photo IDs:",
        missing_zero_id_examples
    )
else:
    print(
        "Every all-zero photo was matched successfully"
    )

Example unmatched all-zero photo IDs: [14254628, 14254630, 14927982, 14927984, 14927985, 14928010, 14928011, 14928045, 14928046, 14928158, 16956865, 16956866, 16956869, 17416192, 17778103, 17778105, 18048157, 18342143, 18349982, 18349983]


In [49]:
recovered_columns = [
    column
    for column in zero_metadata.columns
    if column != "photo_id"
    and column != "source_chunk"
]

zero_metadata_for_merge = (
    zero_metadata.loc[
        zero_metadata["photo_id"].isin(
            zero_record_ids
        ),
        ["photo_id"] + recovered_columns
    ]
    .copy()
)

zero_metadata_for_merge = (
    zero_metadata_for_merge.rename(
        columns={
            column: f"{column}_recovered"
            for column in recovered_columns
        }
    )
)

print(
    "Metadata records retained for merge:",
    f"{len(zero_metadata_for_merge):,}"
)

Metadata records retained for merge: 652,957


In [50]:
zero_records_enriched = zero_records.merge(
    zero_metadata_for_merge,
    on="photo_id",
    how="left",
    validate="one_to_one"
)

print(
    "Zero records after merge:",
    f"{len(zero_records_enriched):,}"
)

Zero records after merge: 653,130


In [51]:
for column in recovered_columns:
    recovered_column = (
        f"{column}_recovered"
    )

    if recovered_column not in zero_records_enriched.columns:
        continue

    if column in zero_records_enriched.columns:
        zero_records_enriched[column] = (
            zero_records_enriched[column]
            .combine_first(
                zero_records_enriched[
                    recovered_column
                ]
            )
        )
    else:
        zero_records_enriched[column] = (
            zero_records_enriched[
                recovered_column
            ]
        )

    zero_records_enriched = (
        zero_records_enriched.drop(
            columns=recovered_column
        )
    )

C:\Users\cheng\AppData\Local\Temp\ipykernel_26296\3125409211.py:12: FutureWarning: The behavior of array concatenation with empty entries is deprecated. In a future version, this will no longer exclude empty items when determining the result dtype. To retain the old behavior, exclude the empty entries before the concat operation.
  .combine_first(
C:\Users\cheng\AppData\Local\Temp\ipykernel_26296\3125409211.py:12: FutureWarning: The behavior of array concatenation with empty entries is deprecated. In a future version, this will no longer exclude empty items when determining the result dtype. To retain the old behavior, exclude the empty entries before the concat operation.
  .combine_first(


In [52]:
zero_records_enriched[
    "zero_metadata_recovered"
] = (
    zero_records_enriched["photo_id"]
    .isin(matched_ids)
)

print(
    "Metadata recovered:",
    f"{zero_records_enriched['zero_metadata_recovered'].sum():,}"
)

print(
    "Metadata not recovered:",
    f"{(~zero_records_enriched['zero_metadata_recovered']).sum():,}"
)

Metadata recovered: 652,957
Metadata not recovered: 173


In [53]:
ZERO_REQUIRED_COLUMNS = [
    "photo_id",
    "sequence_id",
    "site_id",
    "taken",
    "filename",
    "dirname",
    "latitude",
    "longitude"
]

zero_metadata_completeness = pd.DataFrame({
    "column": ZERO_REQUIRED_COLUMNS,
    "non_missing": [
        zero_records_enriched[
            column
        ].notna().sum()
        for column in ZERO_REQUIRED_COLUMNS
    ],
    "missing": [
        zero_records_enriched[
            column
        ].isna().sum()
        for column in ZERO_REQUIRED_COLUMNS
    ]
})

zero_metadata_completeness[
    "percent_complete"
] = (
    zero_metadata_completeness[
        "non_missing"
    ]
    / len(zero_records_enriched)
    * 100
).round(3)

display(zero_metadata_completeness)

,column,non_missing,missing,percent_complete
0,photo_id,653130,0,100.000
1,sequence_id,653130,0,100.000
2,site_id,652957,173,99.974
3,taken,652957,173,99.974
4,filename,653130,0,100.000
5,dirname,653130,0,100.000
6,latitude,652957,173,99.974
7,longitude,652957,173,99.974


In [54]:
zero_pool_eligible = (
    zero_records_enriched.loc[
        zero_records_enriched[
            "zero_metadata_recovered"
        ]
        & zero_records_enriched[
            ZERO_REQUIRED_COLUMNS
        ].notna().all(axis=1)
    ]
    .copy()
)

print(
    "Total all-zero records:",
    f"{len(zero_records_enriched):,}"
)

print(
    "Initial eligible zero records:",
    f"{len(zero_pool_eligible):,}"
)

print(
    "Excluded because required metadata is missing:",
    f"{len(zero_records_enriched) - len(zero_pool_eligible):,}"
)

Total all-zero records: 653,130
Initial eligible zero records: 652,957
Excluded because required metadata is missing: 173


## Prepare zero-confidence sampling variables

Temporal and geographic variables are recreated using the same definitions
used for the positive sampling dataset.

The zero-confidence records are then checked for sufficient diversity across
time window, season, geography, site and sequence before final sampling.

In [55]:
zero_pool_eligible["taken_dt"] = pd.to_datetime(
    zero_pool_eligible["taken"],
    errors="coerce"
)

invalid_year_mask = (
    zero_pool_eligible["taken_dt"].dt.year.lt(2000)
    | zero_pool_eligible["taken_dt"].dt.year.gt(2026)
)

zero_pool_eligible.loc[
    invalid_year_mask,
    "taken_dt"
] = pd.NaT

zero_pool_eligible["year"] = (
    zero_pool_eligible["taken_dt"].dt.year
)

zero_pool_eligible["month"] = (
    zero_pool_eligible["taken_dt"].dt.month
)

zero_pool_eligible["hour"] = (
    zero_pool_eligible["taken_dt"].dt.hour
)

zero_pool_eligible["date"] = (
    zero_pool_eligible["taken_dt"].dt.date
)

season_map = {
    12: "winter",
    1: "winter",
    2: "winter",
    3: "spring",
    4: "spring",
    5: "spring",
    6: "summer",
    7: "summer",
    8: "summer",
    9: "autumn",
    10: "autumn",
    11: "autumn"
}

zero_pool_eligible["season"] = (
    zero_pool_eligible["month"]
    .map(season_map)
)

In [56]:
def assign_time_window(hour):
    if pd.isna(hour):
        return "missing"

    if hour < 5 or hour >= 22:
        return "overnight"

    if hour < 9:
        return "morning_transition"

    if hour < 17:
        return "daytime_core"

    return "evening_transition"


time_window_order = [
    "overnight",
    "morning_transition",
    "daytime_core",
    "evening_transition",
    "missing"
]

zero_pool_eligible["time_window"] = (
    zero_pool_eligible["hour"]
    .apply(assign_time_window)
)

zero_pool_eligible["time_window"] = pd.Categorical(
    zero_pool_eligible["time_window"],
    categories=time_window_order,
    ordered=True
)

zero_pool_eligible["sampling_time_window"] = (
    zero_pool_eligible["time_window"]
)

In [57]:
original_site_geo = (
    sampling_prepared
    .dropna(
        subset=[
            "site_id"
        ]
    )
    .groupby(
        "site_id",
        as_index=False
    )
    .agg(
        latitude=(
            "latitude",
            "first"
        ),
        longitude=(
            "longitude",
            "first"
        )
    )
)

original_uk_core = (
    original_site_geo["latitude"].between(
        49,
        59
    )
    & original_site_geo["longitude"].between(
        -9,
        3
    )
)

_, original_geographic_bins = pd.qcut(
    original_site_geo.loc[
        original_uk_core,
        "latitude"
    ],
    q=4,
    retbins=True
)

print(
    "Original geographic latitude thresholds:",
    original_geographic_bins
)

Original geographic latitude thresholds: [50.12299347 51.3826828  52.38447189 54.77578354 57.27934647]


In [58]:
geographic_labels = [
    "south",
    "south_central",
    "north_central",
    "north"
]

geographic_cut_bins = (
    original_geographic_bins.copy()
)

geographic_cut_bins[0] = -np.inf
geographic_cut_bins[-1] = np.inf

zero_uk_core = (
    zero_pool_eligible["latitude"].between(
        49,
        59
    )
    & zero_pool_eligible["longitude"].between(
        -9,
        3
    )
)

zero_pool_eligible[
    "geographic_group"
] = "outside_uk_core"

zero_core_groups = pd.cut(
    zero_pool_eligible.loc[
        zero_uk_core,
        "latitude"
    ],
    bins=geographic_cut_bins,
    labels=geographic_labels,
    include_lowest=True
)

zero_pool_eligible.loc[
    zero_uk_core,
    "geographic_group"
] = zero_core_groups.astype(
    "object"
)

In [59]:
print(
    "Eligible zero records:",
    f"{len(zero_pool_eligible):,}"
)

print(
    "Missing parsed timestamps:",
    f"{zero_pool_eligible['taken_dt'].isna().sum():,}"
)

print(
    "Missing seasons:",
    f"{zero_pool_eligible['season'].isna().sum():,}"
)

print(
    "Missing time windows:",
    f"{zero_pool_eligible['time_window'].isna().sum():,}"
)

print(
    "Missing geographic groups:",
    f"{zero_pool_eligible['geographic_group'].isna().sum():,}"
)

print(
    "Unique sites:",
    f"{zero_pool_eligible['site_id'].nunique():,}"
)

print(
    "Unique sequences:",
    f"{zero_pool_eligible['sequence_id'].nunique():,}"
)

Eligible zero records: 652,957
Missing parsed timestamps: 0
Missing seasons: 0
Missing time windows: 0
Missing geographic groups: 0
Unique sites: 2,899
Unique sequences: 264,401


In [60]:
zero_time_window_summary = (
    zero_pool_eligible
    .groupby(
        "sampling_time_window",
        observed=True
    )
    .agg(
        records=(
            "photo_id",
            "size"
        ),
        unique_images=(
            "photo_id",
            "nunique"
        ),
        unique_sequences=(
            "sequence_id",
            "nunique"
        ),
        unique_sites=(
            "site_id",
            "nunique"
        ),
        seasons=(
            "season",
            "nunique"
        ),
        geographic_groups=(
            "geographic_group",
            "nunique"
        )
    )
    .reset_index()
)

display(
    zero_time_window_summary
)

,sampling_time_window,records,unique_images,unique_sequences,unique_sites,seasons,geographic_groups
0,overnight,78009,78009,44644,2384,4,5
1,morning_transition,61719,61719,32822,2387,4,5
2,daytime_core,418657,418657,141823,2686,4,5
3,evening_transition,94572,94572,45140,2487,4,5


In [61]:
zero_time_season_table = pd.crosstab(
    zero_pool_eligible[
        "sampling_time_window"
    ],
    zero_pool_eligible[
        "season"
    ],
    margins=True
)

display(
    zero_time_season_table
)

season,autumn,spring,summer,winter,All
sampling_time_window,,,,,
overnight,27171,14320,27389,9129,78009
morning_transition,18733,13774,22959,6253,61719
daytime_core,76323,102292,232259,7783,418657
evening_transition,28910,20802,41124,3736,94572
All,151137,151188,323731,26901,652957


In [62]:
zero_time_geography_table = pd.crosstab(
    zero_pool_eligible[
        "sampling_time_window"
    ],
    zero_pool_eligible[
        "geographic_group"
    ],
    margins=True
)

display(
    zero_time_geography_table
)

geographic_group,north,north_central,outside_uk_core,south,south_central,All
sampling_time_window,,,,,,
overnight,15925,23323,736,21638,16387,78009
morning_transition,9864,19787,525,19184,12359,61719
daytime_core,75402,128568,608,136412,77667,418657
evening_transition,16704,32899,417,25399,19153,94572
All,117895,204577,2286,202633,125566,652957


## Zero-confidence sampling design

The all-zero MegaDetector pool is treated as a shared background stratum because
each manually reviewed image will be labelled for human, animal and vehicle.

The pool is stratified by time window, season and geographic group:

- 4 time windows
- 4 seasons
- 5 geographic groups
- 80 sampling cells

Ten candidate images are selected per cell. Following image-accessibility
checking, five images per cell will be retained, producing a final
zero-confidence sample of 400 images.

In [78]:
ZERO_TIME_WINDOWS = [
    "overnight",
    "morning_transition",
    "daytime_core",
    "evening_transition"
]

ZERO_SEASONS = [
    "winter",
    "spring",
    "summer",
    "autumn"
]

ZERO_GEOGRAPHIC_GROUPS = [
    "south",
    "south_central",
    "north_central",
    "north",
    "outside_uk_core"
]

ZERO_CELL_COLUMNS = [
    "sampling_time_window",
    "season",
    "geographic_group"
]

ZERO_DEFAULT_CANDIDATES_PER_CELL = 10
ZERO_FINAL_PER_CELL = 5

ZERO_MAX_CANDIDATES_PER_SITE_CELL = 3
ZERO_MAX_FINAL_PER_SITE_CELL = 2

ZERO_RANDOM_SEED = 20260729

print(
    "Expected sampling cells:",
    (
        len(ZERO_TIME_WINDOWS)
        * len(ZERO_SEASONS)
        * len(ZERO_GEOGRAPHIC_GROUPS)
    )
)

Expected sampling cells: 80


In [79]:
zero_site_cell_capacity = (
    zero_pool_eligible
    .groupby(
        ZERO_CELL_COLUMNS
        + ["site_id"],
        observed=True
    )
    .agg(
        unique_sequences=(
            "sequence_id",
            "nunique"
        )
    )
    .reset_index()
)

zero_site_cell_capacity[
    "candidate_capacity"
] = (
    zero_site_cell_capacity[
        "unique_sequences"
    ]
    .clip(
        upper=ZERO_MAX_CANDIDATES_PER_SITE_CELL
    )
)

zero_site_cell_capacity[
    "final_capacity"
] = (
    zero_site_cell_capacity[
        "unique_sequences"
    ]
    .clip(
        upper=ZERO_MAX_FINAL_PER_SITE_CELL
    )
)

In [80]:
zero_cell_capacity = (
    zero_site_cell_capacity
    .groupby(
        ZERO_CELL_COLUMNS,
        observed=True
    )
    .agg(
        candidate_capacity=(
            "candidate_capacity",
            "sum"
        ),
        final_capacity=(
            "final_capacity",
            "sum"
        ),
        unique_sites=(
            "site_id",
            "nunique"
        )
    )
    .reset_index()
)

In [81]:
zero_complete_cell_grid = (
    pd.MultiIndex.from_product(
        [
            ZERO_TIME_WINDOWS,
            ZERO_SEASONS,
            ZERO_GEOGRAPHIC_GROUPS
        ],
        names=ZERO_CELL_COLUMNS
    )
    .to_frame(
        index=False
    )
)

zero_cell_targets = (
    zero_complete_cell_grid
    .merge(
        zero_cell_capacity,
        on=ZERO_CELL_COLUMNS,
        how="left",
        validate="one_to_one"
    )
)

capacity_columns = [
    "candidate_capacity",
    "final_capacity",
    "unique_sites"
]

zero_cell_targets[
    capacity_columns
] = (
    zero_cell_targets[
        capacity_columns
    ]
    .fillna(0)
    .astype(int)
)

In [82]:
zero_cell_targets[
    "candidate_target"
] = np.minimum(
    ZERO_DEFAULT_CANDIDATES_PER_CELL,
    zero_cell_targets[
        "candidate_capacity"
    ]
)

zero_cell_targets[
    "final_target"
] = ZERO_FINAL_PER_CELL

zero_cell_targets[
    "final_feasible"
] = (
    zero_cell_targets[
        "final_capacity"
    ]
    >= zero_cell_targets[
        "final_target"
    ]
)

In [83]:
print(
    "Total cells:",
    len(zero_cell_targets)
)

print(
    "Cells feasible for final sample:",
    int(
        zero_cell_targets[
            "final_feasible"
        ].sum()
    )
)

print(
    "Total candidate target:",
    int(
        zero_cell_targets[
            "candidate_target"
        ].sum()
    )
)

print(
    "Total final target:",
    int(
        zero_cell_targets[
            "final_target"
        ].sum()
    )
)

print(
    "Minimum candidate target:",
    int(
        zero_cell_targets[
            "candidate_target"
        ].min()
    )
)

Total cells: 80
Cells feasible for final sample: 80
Total candidate target: 798
Total final target: 400
Minimum candidate target: 8


In [85]:
print(
    zero_cell_targets.loc[
        zero_cell_targets[
            "candidate_target"
        ]
        < ZERO_DEFAULT_CANDIDATES_PER_CELL
    ]
)

   sampling_time_window  season geographic_group  candidate_capacity  \
54         daytime_core  summer  outside_uk_core                   8   

    final_capacity  unique_sites  candidate_target  final_target  \
54               6             4                 8             5   

    final_feasible  
54            True  


In [86]:
rng = np.random.default_rng(
    ZERO_RANDOM_SEED
)

selected_zero_candidates = []

used_zero_photo_ids = set()
used_zero_sequence_ids = set()

ordered_zero_cells = (
    zero_cell_targets
    .sort_values(
        [
            "candidate_capacity",
            "candidate_target"
        ]
    )
    .reset_index(drop=True)
)

for _, cell in ordered_zero_cells.iterrows():
    cell_mask = (
        (
            zero_pool_eligible[
                "sampling_time_window"
            ].astype("string")
            == str(
                cell["sampling_time_window"]
            )
        )
        & (
            zero_pool_eligible[
                "season"
            ].astype("string")
            == str(
                cell["season"]
            )
        )
        & (
            zero_pool_eligible[
                "geographic_group"
            ].astype("string")
            == str(
                cell["geographic_group"]
            )
        )
    )

    cell_pool = (
        zero_pool_eligible.loc[
            cell_mask
        ]
        .copy()
    )

    cell_pool = cell_pool.loc[
        ~cell_pool["photo_id"].isin(
            used_zero_photo_ids
        )
        & ~cell_pool["sequence_id"].isin(
            used_zero_sequence_ids
        )
    ].copy()

    cell_pool["_random_order"] = (
        rng.random(
            len(cell_pool)
        )
    )

    sequence_pool = (
        cell_pool
        .sort_values("_random_order")
        .drop_duplicates(
            subset="sequence_id",
            keep="first"
        )
        .reset_index(drop=True)
    )

    target_count = int(
        cell["candidate_target"]
    )

    selected_in_cell = []
    site_counts = {}

    while len(selected_in_cell) < target_count:
        sequence_pool[
            "_site_count"
        ] = (
            sequence_pool["site_id"]
            .map(site_counts)
            .fillna(0)
            .astype(int)
        )

        available_pool = (
            sequence_pool.loc[
                sequence_pool[
                    "_site_count"
                ]
                < ZERO_MAX_CANDIDATES_PER_SITE_CELL
            ]
            .copy()
        )

        if available_pool.empty:
            break

        minimum_site_count = (
            available_pool[
                "_site_count"
            ].min()
        )

        preferred_pool = (
            available_pool.loc[
                available_pool[
                    "_site_count"
                ]
                == minimum_site_count
            ]
            .copy()
        )

        available_sites = (
            preferred_pool["site_id"]
            .drop_duplicates()
            .tolist()
        )

        selected_site = rng.choice(
            available_sites
        )

        site_pool = (
            preferred_pool.loc[
                preferred_pool["site_id"]
                == selected_site
            ]
            .copy()
        )

        selected_row = site_pool.iloc[
            rng.integers(
                0,
                len(site_pool)
            )
        ].copy()

        selected_in_cell.append(
            selected_row
        )

        selected_photo_id = (
            selected_row["photo_id"]
        )

        selected_sequence_id = (
            selected_row["sequence_id"]
        )

        used_zero_photo_ids.add(
            selected_photo_id
        )

        used_zero_sequence_ids.add(
            selected_sequence_id
        )

        site_counts[selected_site] = (
            site_counts.get(
                selected_site,
                0
            )
            + 1
        )

        sequence_pool = (
            sequence_pool.loc[
                sequence_pool[
                    "sequence_id"
                ]
                != selected_sequence_id
            ]
            .copy()
        )

    if len(selected_in_cell) != target_count:
        raise ValueError(
            "Candidate target not met for "
            f"{cell['sampling_time_window']}, "
            f"{cell['season']}, "
            f"{cell['geographic_group']}. "
            f"Selected {len(selected_in_cell)} "
            f"of {target_count}."
        )

    for rank, selected_row in enumerate(
        selected_in_cell,
        start=1
    ):
        record = selected_row.to_dict()

        record[
            "zero_candidate_rank"
        ] = rank

        record[
            "zero_sampling_cell_id"
        ] = (
            f"{cell['sampling_time_window']}|"
            f"{cell['season']}|"
            f"{cell['geographic_group']}"
        )

        selected_zero_candidates.append(
            record
        )

zero_candidate_records = pd.DataFrame(
    selected_zero_candidates
)

temporary_columns = [
    column
    for column in zero_candidate_records.columns
    if column.startswith("_")
]

zero_candidate_records = (
    zero_candidate_records
    .drop(
        columns=temporary_columns,
        errors="ignore"
    )
    .reset_index(drop=True)
)

zero_candidate_records[
    "zero_candidate_id"
] = [
    f"ZERO_CAND_{index:04d}"
    for index in range(
        1,
        len(zero_candidate_records) + 1
    )
]

print(
    "Selected zero-confidence candidates:",
    f"{len(zero_candidate_records):,}"
)

Selected zero-confidence candidates: 798


In [87]:
zero_candidate_validation = (
    zero_candidate_records
    .groupby(
        ZERO_CELL_COLUMNS,
        observed=True
    )
    .agg(
        selected_candidates=(
            "photo_id",
            "size"
        ),
        unique_images=(
            "photo_id",
            "nunique"
        ),
        unique_sequences=(
            "sequence_id",
            "nunique"
        ),
        unique_sites=(
            "site_id",
            "nunique"
        ),
        maximum_site_repetition=(
            "site_id",
            lambda values: (
                values.value_counts().max()
            )
        )
    )
    .reset_index()
    .merge(
        zero_cell_targets[
            ZERO_CELL_COLUMNS
            + ["candidate_target"]
        ],
        on=ZERO_CELL_COLUMNS,
        how="left",
        validate="one_to_one"
    )
)

zero_candidate_validation[
    "target_met"
] = (
    zero_candidate_validation[
        "selected_candidates"
    ]
    == zero_candidate_validation[
        "candidate_target"
    ]
)

print(
    "Candidate records:",
    f"{len(zero_candidate_records):,}"
)

print(
    "Candidate cells:",
    len(zero_candidate_validation)
)

print(
    "Cells meeting target:",
    int(
        zero_candidate_validation[
            "target_met"
        ].sum()
    )
)

print(
    "Duplicated photo IDs:",
    int(
        zero_candidate_records[
            "photo_id"
        ].duplicated().sum()
    )
)

print(
    "Duplicated sequence IDs:",
    int(
        zero_candidate_records[
            "sequence_id"
        ].duplicated().sum()
    )
)

print(
    "Maximum site repetition within a cell:",
    int(
        zero_candidate_validation[
            "maximum_site_repetition"
        ].max()
    )
)

Candidate records: 798
Candidate cells: 80
Cells meeting target: 80
Duplicated photo IDs: 0
Duplicated sequence IDs: 0
Maximum site repetition within a cell: 3


In [88]:
ZERO_CANDIDATE_RECORDS_PATH = (
    OUTPUT_DIR
    / "zero_confidence_candidate_records.csv"
)

zero_candidate_records.to_csv(
    ZERO_CANDIDATE_RECORDS_PATH,
    index=False
)

print(
    "Saved candidate records:",
    ZERO_CANDIDATE_RECORDS_PATH
)

print(
    "Candidate records:",
    f"{len(zero_candidate_records):,}"
)

Saved candidate records: sampling_outputs\zero_confidence_candidate_records.csv
Candidate records: 798


In [89]:
S3_BASE_URL = (
    "https://mammalweb.s3-eu-west-1.amazonaws.com"
)

LOCAL_IMAGE_PREFIX = (
    "/var/www/html/biodivimages/"
)


def build_s3_url(dirname, filename):
    directory = (
        str(dirname)
        .strip()
        .replace("\\", "/")
    )

    image_filename = (
        str(filename)
        .strip()
        .lstrip("/")
    )

    if directory.startswith(
        LOCAL_IMAGE_PREFIX
    ):
        directory = directory[
            len(LOCAL_IMAGE_PREFIX):
        ]

    directory = directory.strip("/")

    return (
        f"{S3_BASE_URL}/"
        f"{directory}/"
        f"{image_filename}"
    )


zero_candidate_records[
    "s3_url"
] = zero_candidate_records.apply(
    lambda row: build_s3_url(
        row["dirname"],
        row["filename"]
    ),
    axis=1
)

In [90]:
ZERO_CANDIDATE_QUEUE_PATH = (
    OUTPUT_DIR
    / "zero_confidence_candidate_image_queue.csv"
)

zero_candidate_queue_columns = [
    "zero_candidate_id",
    "zero_sampling_cell_id",
    "zero_candidate_rank",
    "photo_id",
    "sequence_id",
    "sequence_num",
    "site_id",
    "person_id",
    "sampling_time_window",
    "season",
    "geographic_group",
    "taken",
    "dirname",
    "filename",
    "s3_url"
]

available_queue_columns = [
    column
    for column
    in zero_candidate_queue_columns
    if column
    in zero_candidate_records.columns
]

zero_candidate_image_queue = (
    zero_candidate_records[
        available_queue_columns
    ]
    .copy()
)

zero_candidate_image_queue.to_csv(
    ZERO_CANDIDATE_QUEUE_PATH,
    index=False
)

print(
    "Candidate queue records:",
    f"{len(zero_candidate_image_queue):,}"
)

print(
    "Unique photo IDs:",
    f"{zero_candidate_image_queue['photo_id'].nunique():,}"
)

print(
    "Unique sequences:",
    f"{zero_candidate_image_queue['sequence_id'].nunique():,}"
)

print(
    "Unique S3 URLs:",
    f"{zero_candidate_image_queue['s3_url'].nunique():,}"
)

print(
    "Missing S3 URLs:",
    f"{zero_candidate_image_queue['s3_url'].isna().sum():,}"
)

print(
    "Saved candidate image queue:",
    ZERO_CANDIDATE_QUEUE_PATH
)

Candidate queue records: 798
Unique photo IDs: 798
Unique sequences: 798
Unique S3 URLs: 798
Missing S3 URLs: 0
Saved candidate image queue: sampling_outputs\zero_confidence_candidate_image_queue.csv


## Zero-confidence image accessibility

The zero-confidence candidate URLs are checked before final sampling. Unlike the
positive-confidence sample, bounding-box dimensions are not required because
these records contain no stored target-class detection.

In [91]:
from concurrent.futures import ThreadPoolExecutor, as_completed

import requests
from tqdm.auto import tqdm

C:\Users\cheng\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [92]:
def check_zero_candidate_url(record):
    candidate_id = record["zero_candidate_id"]
    s3_url = record["s3_url"]

    try:
        response = requests.get(
            s3_url,
            headers={
                "Range": "bytes=0-0"
            },
            stream=True,
            allow_redirects=True,
            timeout=(5, 20)
        )

        status_code = response.status_code
        content_type = response.headers.get(
            "Content-Type",
            ""
        )

        response.close()

        return {
            "zero_candidate_id": candidate_id,
            "image_accessible": (
                status_code in [200, 206]
            ),
            "http_status": status_code,
            "content_type": content_type,
            "access_error": None
        }

    except requests.RequestException as error:
        return {
            "zero_candidate_id": candidate_id,
            "image_accessible": False,
            "http_status": None,
            "content_type": None,
            "access_error": str(error)
        }

In [93]:
zero_accessibility_results = []

candidate_records_for_check = (
    zero_candidate_image_queue[
        [
            "zero_candidate_id",
            "s3_url"
        ]
    ]
    .to_dict("records")
)

with ThreadPoolExecutor(
    max_workers=16
) as executor:
    futures = [
        executor.submit(
            check_zero_candidate_url,
            record
        )
        for record
        in candidate_records_for_check
    ]

    for future in tqdm(
        as_completed(futures),
        total=len(futures),
        desc="Checking zero-confidence images"
    ):
        zero_accessibility_results.append(
            future.result()
        )

zero_accessibility = pd.DataFrame(
    zero_accessibility_results
)

Checking zero-confidence images: 100%|██████████| 798/798 [01:14<00:00, 10.78it/s]


In [94]:
zero_candidate_image_queue = (
    zero_candidate_image_queue
    .merge(
        zero_accessibility,
        on="zero_candidate_id",
        how="left",
        validate="one_to_one"
    )
)

print(
    "Candidate images:",
    f"{len(zero_candidate_image_queue):,}"
)

print(
    "Accessible images:",
    f"{zero_candidate_image_queue['image_accessible'].sum():,}"
)

print(
    "Inaccessible images:",
    f"{(~zero_candidate_image_queue['image_accessible']).sum():,}"
)

display(
    zero_candidate_image_queue[
        "http_status"
    ]
    .value_counts(
        dropna=False
    )
    .rename_axis("http_status")
    .reset_index(name="records")
)

Candidate images: 798
Accessible images: 798
Inaccessible images: 0


,http_status,records
0,206,798


In [95]:
zero_accessible_cell_summary = (
    zero_candidate_image_queue
    .groupby(
        "zero_sampling_cell_id",
        observed=True
    )
    .agg(
        candidate_images=(
            "photo_id",
            "size"
        ),
        accessible_images=(
            "image_accessible",
            "sum"
        )
    )
    .reset_index()
)

print(
    "Sampling cells:",
    len(zero_accessible_cell_summary)
)

print(
    "Cells with at least five accessible images:",
    int(
        zero_accessible_cell_summary[
            "accessible_images"
        ].ge(5).sum()
    )
)

print(
    "Minimum accessible images in one cell:",
    int(
        zero_accessible_cell_summary[
            "accessible_images"
        ].min()
    )
)

display(
    zero_accessible_cell_summary.loc[
        zero_accessible_cell_summary[
            "accessible_images"
        ] < 5
    ]
)

Sampling cells: 80
Cells with at least five accessible images: 80
Minimum accessible images in one cell: 8


,zero_sampling_cell_id,candidate_images,accessible_images


In [96]:
final_zero_records = []

accessible_zero_candidates = (
    zero_candidate_image_queue.loc[
        zero_candidate_image_queue[
            "image_accessible"
        ]
    ]
    .copy()
)

for cell_id, cell_pool in (
    accessible_zero_candidates
    .groupby(
        "zero_sampling_cell_id",
        observed=True
    )
):
    cell_pool = (
        cell_pool
        .sort_values(
            "zero_candidate_rank"
        )
        .copy()
    )

    selected_in_cell = []
    site_counts = {}

    while len(selected_in_cell) < ZERO_FINAL_PER_CELL:
        cell_pool[
            "_site_selection_count"
        ] = (
            cell_pool["site_id"]
            .map(site_counts)
            .fillna(0)
            .astype(int)
        )

        available_pool = (
            cell_pool.loc[
                cell_pool[
                    "_site_selection_count"
                ]
                < ZERO_MAX_FINAL_PER_SITE_CELL
            ]
            .copy()
        )

        if available_pool.empty:
            break

        minimum_site_count = (
            available_pool[
                "_site_selection_count"
            ].min()
        )

        preferred_pool = (
            available_pool.loc[
                available_pool[
                    "_site_selection_count"
                ]
                == minimum_site_count
            ]
            .sort_values(
                "zero_candidate_rank"
            )
        )

        selected_row = (
            preferred_pool.iloc[0]
            .copy()
        )

        selected_in_cell.append(
            selected_row
        )

        selected_site = (
            selected_row["site_id"]
        )

        site_counts[selected_site] = (
            site_counts.get(
                selected_site,
                0
            )
            + 1
        )

        cell_pool = (
            cell_pool.loc[
                cell_pool[
                    "zero_candidate_id"
                ]
                != selected_row[
                    "zero_candidate_id"
                ]
            ]
            .copy()
        )

    if len(selected_in_cell) != ZERO_FINAL_PER_CELL:
        raise ValueError(
            f"Could not select five final images for {cell_id}. "
            f"Selected {len(selected_in_cell)}."
        )

    final_zero_records.extend(
        selected_in_cell
    )

In [97]:
final_zero_sample = pd.DataFrame(
    final_zero_records
)

temporary_columns = [
    column
    for column in final_zero_sample.columns
    if column.startswith("_")
]

final_zero_sample = (
    final_zero_sample
    .drop(
        columns=temporary_columns,
        errors="ignore"
    )
    .sort_values(
        [
            "zero_sampling_cell_id",
            "zero_candidate_rank"
        ]
    )
    .reset_index(drop=True)
)

final_zero_sample[
    "zero_final_id"
] = [
    f"ZERO_FINAL_{index:04d}"
    for index in range(
        1,
        len(final_zero_sample) + 1
    )
]

final_zero_sample[
    "sample_type"
] = "zero_confidence"

print(
    "Final zero-confidence images:",
    f"{len(final_zero_sample):,}"
)

Final zero-confidence images: 400


In [98]:
final_zero_validation = (
    final_zero_sample
    .groupby(
        "zero_sampling_cell_id",
        observed=True
    )
    .agg(
        selected_images=(
            "photo_id",
            "size"
        ),
        unique_images=(
            "photo_id",
            "nunique"
        ),
        unique_sequences=(
            "sequence_id",
            "nunique"
        ),
        unique_sites=(
            "site_id",
            "nunique"
        ),
        maximum_site_repetition=(
            "site_id",
            lambda values: (
                values.value_counts().max()
            )
        )
    )
    .reset_index()
)

print(
    "Final images:",
    f"{len(final_zero_sample):,}"
)

print(
    "Final sampling cells:",
    len(final_zero_validation)
)

print(
    "Cells containing exactly five images:",
    int(
        final_zero_validation[
            "selected_images"
        ].eq(5).sum()
    )
)

print(
    "Duplicated photo IDs:",
    int(
        final_zero_sample[
            "photo_id"
        ].duplicated().sum()
    )
)

print(
    "Duplicated sequence IDs:",
    int(
        final_zero_sample[
            "sequence_id"
        ].duplicated().sum()
    )
)

print(
    "Maximum site repetition within a cell:",
    int(
        final_zero_validation[
            "maximum_site_repetition"
        ].max()
    )
)

print(
    "Inaccessible final images:",
    int(
        (
            ~final_zero_sample[
                "image_accessible"
            ]
        ).sum()
    )
)

Final images: 400
Final sampling cells: 80
Cells containing exactly five images: 80
Duplicated photo IDs: 0
Duplicated sequence IDs: 0
Maximum site repetition within a cell: 2
Inaccessible final images: 0


In [99]:
FINAL_ZERO_SAMPLE_PATH = (
    OUTPUT_DIR
    / "final_zero_confidence_image_queue.csv"
)

final_zero_sample.to_csv(
    FINAL_ZERO_SAMPLE_PATH,
    index=False
)

print(
    "Saved final zero-confidence sample:",
    FINAL_ZERO_SAMPLE_PATH
)

Saved final zero-confidence sample: sampling_outputs\final_zero_confidence_image_queue.csv


## Combined blinded manual-annotation queue

The final positive-confidence and zero-confidence image samples are combined
into one image-level queue.

The queue is randomly ordered and the sampling stratum is hidden during manual
annotation to reduce expectation bias. A separate master key retains the
sampling information for later analysis.

In [100]:
positive_for_annotation = (
    final_positive_image_queue.copy()
)

zero_for_annotation = (
    final_zero_sample.copy()
)

positive_for_annotation[
    "sample_type"
] = "positive_confidence"

positive_for_annotation[
    "source_sample_id"
] = positive_for_annotation[
    "positive_image_queue_id"
]

positive_for_annotation[
    "annotation_image_url"
] = positive_for_annotation[
    "image_url"
]

positive_for_annotation[
    "evaluation_target_classes"
] = positive_for_annotation[
    "positive_target_classes"
]

positive_for_annotation[
    "source_sampling_cell_ids"
] = positive_for_annotation[
    "positive_sampling_cell_ids"
]

In [101]:
zero_for_annotation[
    "sample_type"
] = "zero_confidence"

zero_for_annotation[
    "source_sample_id"
] = zero_for_annotation[
    "zero_final_id"
]

zero_for_annotation[
    "annotation_image_url"
] = zero_for_annotation[
    "s3_url"
]

zero_for_annotation[
    "evaluation_target_classes"
] = "human|animal|vehicle"

zero_for_annotation[
    "source_sampling_cell_ids"
] = zero_for_annotation[
    "zero_sampling_cell_id"
]

for target_class in TARGET_CLASSES:
    zero_for_annotation[
        f"selected_for_{target_class}"
    ] = True

In [102]:
MASTER_QUEUE_COLUMNS = [
    "sample_type",
    "source_sample_id",
    "photo_id",
    "sequence_id",
    "sequence_num",
    "site_id",
    "person_id",
    "taken",
    "filename",
    "dirname",
    "annotation_image_url",
    "sampling_time_window",
    "season",
    "geographic_group",
    "evaluation_target_classes",
    "source_sampling_cell_ids",
    "selected_for_human",
    "selected_for_animal",
    "selected_for_vehicle"
]

for dataframe in [
    positive_for_annotation,
    zero_for_annotation
]:
    for column in MASTER_QUEUE_COLUMNS:
        if column not in dataframe.columns:
            dataframe[column] = pd.NA

master_annotation_key = pd.concat(
    [
        positive_for_annotation[
            MASTER_QUEUE_COLUMNS
        ],
        zero_for_annotation[
            MASTER_QUEUE_COLUMNS
        ]
    ],
    ignore_index=True
)

C:\Users\cheng\AppData\Local\Temp\ipykernel_26296\42404406.py:31: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  master_annotation_key = pd.concat(


In [103]:
positive_photo_ids = set(
    positive_for_annotation["photo_id"]
)

zero_photo_ids = set(
    zero_for_annotation["photo_id"]
)

overlapping_photo_ids = (
    positive_photo_ids
    & zero_photo_ids
)

print(
    "Positive images:",
    f"{len(positive_for_annotation):,}"
)

print(
    "Zero-confidence images:",
    f"{len(zero_for_annotation):,}"
)

print(
    "Combined images:",
    f"{len(master_annotation_key):,}"
)

print(
    "Overlapping positive and zero images:",
    len(overlapping_photo_ids)
)

print(
    "Duplicated photo IDs:",
    int(
        master_annotation_key[
            "photo_id"
        ].duplicated().sum()
    )
)

Positive images: 1,677
Zero-confidence images: 400
Combined images: 2,077
Overlapping positive and zero images: 0
Duplicated photo IDs: 0


In [104]:
ANNOTATION_RANDOM_SEED = 20260729

master_annotation_key = (
    master_annotation_key
    .sample(
        frac=1,
        random_state=ANNOTATION_RANDOM_SEED
    )
    .reset_index(drop=True)
)

master_annotation_key[
    "annotation_queue_id"
] = [
    f"ANNOT_{index:04d}"
    for index in range(
        1,
        len(master_annotation_key) + 1
    )
]

In [105]:
blinded_annotation_queue = (
    master_annotation_key[
        [
            "annotation_queue_id",
            "photo_id",
            "sequence_id",
            "site_id",
            "filename",
            "dirname",
            "annotation_image_url"
        ]
    ]
    .copy()
)

blinded_annotation_queue[
    "annotation_status"
] = "pending"

blinded_annotation_queue[
    "manual_human_present"
] = pd.NA

blinded_annotation_queue[
    "manual_animal_present"
] = pd.NA

blinded_annotation_queue[
    "manual_vehicle_present"
] = pd.NA

blinded_annotation_queue[
    "human_recognisable"
] = pd.NA

blinded_annotation_queue[
    "privacy_risk"
] = pd.NA

blinded_annotation_queue[
    "safeguarding_risk"
] = pd.NA

blinded_annotation_queue[
    "annotation_quality"
] = pd.NA

blinded_annotation_queue[
    "notes"
] = pd.NA

In [106]:
MASTER_ANNOTATION_KEY_PATH = (
    OUTPUT_DIR
    / "master_annotation_sampling_key.csv"
)

BLINDED_ANNOTATION_QUEUE_PATH = (
    OUTPUT_DIR
    / "blinded_manual_annotation_queue.csv"
)

master_annotation_key.to_csv(
    MASTER_ANNOTATION_KEY_PATH,
    index=False
)

blinded_annotation_queue.to_csv(
    BLINDED_ANNOTATION_QUEUE_PATH,
    index=False
)

print(
    "Saved master sampling key:",
    MASTER_ANNOTATION_KEY_PATH
)

print(
    "Saved blinded annotation queue:",
    BLINDED_ANNOTATION_QUEUE_PATH
)

print(
    "Images awaiting annotation:",
    f"{len(blinded_annotation_queue):,}"
)

Saved master sampling key: sampling_outputs\master_annotation_sampling_key.csv
Saved blinded annotation queue: sampling_outputs\blinded_manual_annotation_queue.csv
Images awaiting annotation: 2,077
